# Training Model Deteksi APD

Di notebook ini kita melatih YOLO11 untuk mengenali helm, rompi, masker, dan pekerja yang tidak memakainya.

Sebelum mulai, aktifkan GPU: menu `Runtime` > `Change runtime type` > pilih `T4 GPU` > `Save`.

Jalankan sel satu per satu dari atas. Klik tombol play di kiri sel, atau tekan `Shift + Enter`.

## Langkah 1 - Install library

In [ ]:
!pip install -q ultralytics kagglehub

## Langkah 2 - Download dataset dari Kaggle

In [ ]:
import kagglehub

folder = kagglehub.dataset_download("snehilsanyal/construction-site-safety-image-dataset-roboflow")
dataset = folder + "/css-data"
print(dataset)

## Langkah 3 - Buat file konfigurasi dataset

Isinya lokasi folder gambar dan daftar nama kelas.

In [ ]:
kelas = ["Hardhat", "Mask", "NO-Hardhat", "NO-Mask", "NO-Safety Vest",
         "Person", "Safety Cone", "Safety Vest", "machinery", "vehicle"]

isi = f"""
train: {dataset}/train/images
val: {dataset}/valid/images
test: {dataset}/test/images
names: {kelas}
"""

with open("data.yaml", "w") as f:
    f.write(isi)

print(isi)

## Langkah 4 - Lihat contoh gambar

In [ ]:
import os
from IPython.display import Image, display

contoh = os.listdir(dataset + "/train/images")[:3]
for nama in contoh:
    display(Image(dataset + "/train/images/" + nama, width=400))

## Langkah 5 - Training model

`epochs=30` artinya model membaca seluruh gambar training sebanyak 30 kali. Dengan GPU T4, prosesnya sekitar 30 sampai 40 menit.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")
model.train(data="data.yaml", epochs=30, imgsz=640)

## Langkah 6 - Lihat hasil training

Di `results.png`, grafik loss sebaiknya turun dan grafik mAP naik. Confusion matrix menunjukkan kelas yang sering tertukar.

In [ ]:
hasil = model.trainer.save_dir
display(Image(f"{hasil}/results.png", width=900))
display(Image(f"{hasil}/confusion_matrix_normalized.png", width=700))

## Langkah 7 - Uji model pada gambar test

In [ ]:
model = YOLO(f"{hasil}/weights/best.pt")

metrics = model.val(data="data.yaml", split="test")
print("mAP50 :", round(metrics.box.map50, 3))

gambar = dataset + "/test/images/" + os.listdir(dataset + "/test/images")[0]
hasil_prediksi = model(gambar)
hasil_prediksi[0].save("prediksi.jpg")
display(Image("prediksi.jpg", width=600))

## Langkah 8 - Download model

Simpan `best.pt` ke folder `models/` di proyek Anda. Web deteksi membaca model dari sana.

In [ ]:
from google.colab import files

files.download(f"{hasil}/weights/best.pt")